In [107]:
#Data Cleaning and Validation

#settings
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RAW_PATH = "../data/raw/atp_matches_2024.csv"


In [108]:
#Load data
df = pd.read_csv(RAW_PATH)
df.head()


,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,winner_name,winner_hand,winner_ht,winner_ioc,winner_age,loser_id,loser_seed,loser_entry,loser_name,loser_hand,loser_ht,loser_ioc,loser_age,score,best_of,round,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2024-0339,Brisbane,Hard,32,A,20240101,300,105777,2.0,NaN,Grigor Dimitrov,R,191.0,BUL,32.6,208029,1.0,NaN,Holger Rune,R,188.0,DEN,20.6,7-6(5) 6-4,3,F,136.0,8.0,2.0,74.0,52.0,40.0,13.0,11.0,3.0,3.0,9.0,3.0,95.0,58.0,44.0,16.0,11.0,8.0,9.0,14.0,2570.0,8.0,3660.0
1,2024-0339,Brisbane,Hard,32,A,20240101,299,208029,1.0,NaN,Holger Rune,R,188.0,DEN,20.6,126128,NaN,NaN,Roman Safiullin,R,185.0,RUS,26.4,6-4 7-6(0),3,SF,97.0,7.0,4.0,72.0,48.0,39.0,11.0,11.0,1.0,2.0,5.0,5.0,66.0,35.0,31.0,10.0,11.0,5.0,7.0,8.0,3660.0,39.0,1122.0
2,2024-0339,Brisbane,Hard,32,A,20240101,298,105777,2.0,NaN,Grigor Dimitrov,R,191.0,BUL,32.6,111442,NaN,NaN,Jordan Thompson,R,183.0,AUS,29.7,6-3 7-5,3,SF,109.0,10.0,3.0,67.0,45.0,39.0,10.0,11.0,6.0,6.0,5.0,1.0,62.0,39.0,24.0,14.0,10.0,5.0,7.0,14.0,2570.0,55.0,902.0
3,2024-0339,Brisbane,Hard,32,A,20240101,297,208029,1.0,NaN,Holger Rune,R,188.0,DEN,20.6,105902,NaN,Q,James Duckworth,R,183.0,AUS,31.9,6-2 7-6(6),3,QF,105.0,13.0,0.0,65.0,36.0,31.0,17.0,10.0,1.0,1.0,5.0,2.0,77.0,51.0,31.0,16.0,10.0,3.0,5.0,8.0,3660.0,116.0,573.0
4,2024-0339,Brisbane,Hard,32,A,20240101,296,126128,NaN,NaN,Roman Safiullin,R,185.0,RUS,26.4,208286,NaN,NaN,Matteo Arnaldi,R,185.0,ITA,22.8,7-6(4) 6-2,3,QF,120.0,9.0,3.0,73.0,43.0,36.0,14.0,10.0,2.0,3.0,3.0,2.0,69.0,37.0,27.0,16.0,10.0,5.0,8.0,39.0,1122.0,44.0,1021.0


In [109]:
#step 1 : Header Cleaning
def normalise_headers(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace("-", "_")
    return df

In [110]:
#step 2 : Type Conversion

df.info()

#tourney date int -> datetime
def parse_dates(df):
    df["tourney_date"] = pd.to_datetime( df["tourney_date"], format="%Y%m%d", errors="coerce" )
    return df

#serve stats -> numeric
def convert_numeric(df):
    serve_cols = [c for c in df.columns if c.startswith(("w_", "l_"))]
    df[serve_cols] = df[serve_cols].apply(pd.to_numeric, errors="coerce")
    return df

#all rankings and seedings -> integers
def clean_ranks_and_seeds(df):
    seed_cols = ["winner_seed", "loser_seed"]
    rank_cols = ["winner_rank", "loser_rank", "winner_rank_points", "loser_rank_points"]
    df[seed_cols] = df[seed_cols].astype("Int64")
    df[rank_cols] = df[rank_cols].astype("Int64")
    return df




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3076 entries, 0 to 3075
Data columns (total 49 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   tourney_id          3076 non-null   object 
 1   tourney_name        3076 non-null   object 
 2   surface             3076 non-null   object 
 3   draw_size           3076 non-null   int64  
 4   tourney_level       3076 non-null   object 
 5   tourney_date        3076 non-null   int64  
 6   match_num           3076 non-null   int64  
 7   winner_id           3076 non-null   int64  
 8   winner_seed         1294 non-null   float64
 9   winner_entry        477 non-null    object 
 10  winner_name         3076 non-null   object 
 11  winner_hand         3076 non-null   object 
 12  winner_ht           3057 non-null   float64
 13  winner_ioc          3076 non-null   object 
 14  winner_age          3075 non-null   float64
 15  loser_id            3076 non-null   int64  
 16  loser_

In [111]:
#step 3 : Value Mapping (help model)
def map_hands(df):
    hand_map = {"R": "Right", "L": "Left", "U": "Unknown"}
    df["winner_hand"] = df["winner_hand"].map(hand_map)
    df["loser_hand"] = df["loser_hand"].map(hand_map)
    return df


In [112]:
#step 4 : category normalisation
def clean_categories(df):
    df["surface"] = df["surface"].str.title()
    df["round"] = df["round"].str.upper()
    return df


In [113]:
#step 5 : physical value validation (no crazy values)
def validate_physical_stats(df):
    df = df[df["winner_ht"].between(140, 220) | df["winner_ht"].isna()]
    df = df[df["loser_ht"].between(140, 220) | df["loser_ht"].isna()]
    df = df[df["winner_age"].between(14, 45) | df["winner_age"].isna()]
    df = df[df["loser_age"].between(14, 45) | df["loser_age"].isna()]
    return df


def validate_surface(df):
    valid = {"Hard", "Clay", "Grass"}
    bad = df[~df["surface"].isin(valid)] #bad = all surfaces not in above list
    if len(bad) > 0:
        print("Warning: unexpected surface values found:")
        print(bad["surface"].unique())
    return df

def validate_round(df):
    valid = {"R128","R64","R32","R16","QF","SF","F","RR"}
    bad = df[~df["round"].isin(valid)]
    if len(bad) > 0:
        print("Warning: unexpected round values found:")
        print(bad["round"].unique())
    return df

def validate_best_of(df):
    bad = df[~df["best_of"].isin([3,5])]
    if len(bad) > 0:
        print("Warning: unexpected best_of values found:")
        print(bad["best_of"].unique())
    return df

def validate_minutes(df):
    bad = df[(df["minutes"] < 20) & df["minutes"].notna()] # length not under 20 unless null
    if len(bad) > 0:
        print("Warning: unusually short matches detected.")
    bad = df[(df["minutes"] > 400)]
    if len(bad) > 0:
        print("Warning: unusually long matches detected.")
    return df




In [114]:
#step 6 : remove all matches that are not atp events
print (df["tourney_name"].unique())
print (df["tourney_level"].unique())
def filter_atp_events(df):
    allowed_levels = {"A", "G", "M", "F"}
    df = df[df["tourney_level"].isin(allowed_levels)]

    banned_keywords = [
        "United Cup",
        "Laver Cup",
        "Next Gen Finals",
        "Olympics",
        "Davis Cup",
        "Exhibition",
        "Kooyong"
    ]

    pattern = "|".join(banned_keywords)
    df = df[~df["tourney_name"].str.contains(pattern, case=False, na=False)]

    return df


['Brisbane' 'United Cup' 'Hong Kong' 'Adelaide' 'Auckland'
 'Australian Open' 'Montpellier' 'Cordoba' 'Dallas' 'Marseille'
 'Delray Beach' 'Buenos Aires' 'Rotterdam' 'Doha' 'Los Cabos'
 'Rio De Janeiro' 'Acapulco' 'Dubai' 'Santiago' 'Indian Wells Masters'
 'Miami Masters' 'Estoril' 'Houston' 'Marrakech' 'Monte Carlo Masters'
 'Barcelona' 'Munich' 'Bucharest' 'Madrid Masters' 'Rome Masters' 'Geneva'
 'Lyon' 'Roland Garros' 's Hertogenbosch' 'Stuttgart' 'Halle'
 "Queen's Club" 'Mallorca' 'Eastbourne' 'Wimbledon' 'Gstaad' 'Hamburg'
 'Bastad' 'Newport' 'Atlanta' 'Kitzbuhel' 'Umag' 'Washington'
 'Paris Olympics' 'Canada Masters' 'Cincinnati Masters' 'Winston-Salem'
 'Us Open' 'Chengdu' 'Hangzhou' 'Laver Cup' 'Tokyo' 'Beijing'
 'Shanghai Masters' 'Almaty' 'Antwerp' 'Stockholm' 'Basel' 'Vienna'
 'Paris Masters' 'Belgrade' 'Metz' 'Tour Finals' 'Next Gen Finals'
 'Davis Cup Finals RR: BEL vs BRA' 'Davis Cup Finals RR: ITA vs BEL'
 'Davis Cup Finals RR: ITA vs BRA' 'Davis Cup Finals RR: ITA vs N

In [115]:
#final step
def clean_matches(df):
    df = (
        df.pipe(normalise_headers)
        .pipe(parse_dates)
        .pipe(convert_numeric)
        .pipe(clean_ranks_and_seeds)
        .pipe(map_hands)
        .pipe(clean_categories)
        .pipe(validate_physical_stats)
        .pipe(validate_surface)
        .pipe(validate_round)
        .pipe(validate_best_of)
        .pipe(validate_minutes)
        .pipe(filter_atp_events)
    )
    return df




df_clean = clean_matches(df)
df_clean.info()
df.head()
print(df_clean.loc[df["round"] == "BR", ["tourney_id", "tourney_name", "round", "winner_name", "loser_name"]])
print (df_clean["tourney_name"].unique())
print (df_clean["tourney_level"].unique())


['BR']
<class 'pandas.core.frame.DataFrame'>
Index: 2703 entries, 0 to 2800
Data columns (total 49 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   tourney_id          2703 non-null   object        
 1   tourney_name        2703 non-null   object        
 2   surface             2703 non-null   object        
 3   draw_size           2703 non-null   int64         
 4   tourney_level       2703 non-null   object        
 5   tourney_date        2703 non-null   datetime64[ns]
 6   match_num           2703 non-null   int64         
 7   winner_id           2703 non-null   int64         
 8   winner_seed         1240 non-null   Int64         
 9   winner_entry        473 non-null    object        
 10  winner_name         2703 non-null   object        
 11  winner_hand         2703 non-null   object        
 12  winner_ht           2702 non-null   float64       
 13  winner_ioc          2703 non-null   object    